# Workplace Unused Benefits Agent — Langfuse + Jev

A hands-on evaluation lab for a fictional employee-benefits agent. It covers tracing, an 18-case golden dataset, structured decisions, balanced human review in Langfuse, prompt improvement, regression checks, and separate evaluator calibration with Jev.

The agent receives employee and claim details together and returns one decision: `eligible`, `not_eligible`, or `needs_review`.


## 1. Install dependencies

Install the packages used for the LLM and Langfuse.


In [7]:
%pip install -q -U "langfuse>=4.8.1,<5" "langchain>=1,<2" "langchain-core>=0.3,<2" "langchain-openai>=0.2,<2" python-dotenv pandas scikit-learn requests tqdm ipykernel


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import json
import requests
import pandas as pd
from collections import Counter
from typing import Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langfuse import get_client, observe, propagate_attributes
from langfuse.langchain import CallbackHandler
try:
    from langfuse import Evaluation
except ImportError:
    from langfuse.experiment import Evaluation


## 2. Load API keys

Add your OpenAI key and Langfuse project keys to `.env`. `LANGFUSE_HOST` is your Langfuse base URL, based of region.


In [9]:
load_dotenv(override=True)
required = ["OPENAI_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_HOST"]
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {', '.join(missing)}. Fill in your .env file first.")
print("Required API settings found.")

BASE_URL = os.environ["LANGFUSE_HOST"].rstrip("/")
AUTH = (os.environ["LANGFUSE_PUBLIC_KEY"], os.environ["LANGFUSE_SECRET_KEY"])


Required API settings found.


## Run configuration

Defining labels for all datasets, queues, and experiments needed. A fresh experiment label is generated for each rehearsal. Annotation Queue upon creating in Langfuse can be named here.

In [10]:
from datetime import datetime, timezone
RUN_LABEL = datetime.now(timezone.utc).strftime("benefits-%Y%m%d-%H%M%S")
DATASET_NAME = "unused-benefits-golden-v1"
ANNOTATION_QUEUE_NAME = "unused-benefits-v1-human-review"
SCORE_CONFIG_NAME = "decision_correct"
HUMAN_SCORE_CONFIG_NAME = "human_decision"
JEV_SCORE_NAME = "based_employee_details"
EXPERIMENT_V1_NAME = f"{RUN_LABEL}-v1"
EXPERIMENT_V2_NAME = f"{RUN_LABEL}-v2"
REGRESSION_DATASET_NAME = f"{RUN_LABEL}-regression"
REVIEW_DATASET_NAME = f"{RUN_LABEL}-review"
print("Run label:", RUN_LABEL)
print("Review queue:", ANNOTATION_QUEUE_NAME)

Run label: benefits-20261020-112631
Review queue: unused-benefits-v1-human-review


## 3. Connect Langfuse and set up tracing

Traces let us inspect the claim input, model call, structured decision, and timing in Langfuse, ensuring valid integration.


In [11]:
langfuse = get_client()
if not langfuse.auth_check():
    raise RuntimeError("Langfuse authentication failed. Check the keys and LANGFUSE_HOST in .env.")
print("Connected to Langfuse.")


Connected to Langfuse.


## 4. Define the policy text

These short fictional policies are provided directly to the LLM as context. The agent interprets them and based on this, assess claims and eligibility of employees.


In [12]:
POLICY_PASSAGES = [
    {
        "policy_id": "LEARNING-ELIGIBILITY-v1", "benefit": "learning", "section": "eligibility",
        "text": "Learning: Full-time employees in IN or US are eligible. The course must relate to the employee's job or agreed development plan; hobby courses are excluded."
    },
    {
        "policy_id": "LEARNING-CLAIM-v1", "benefit": "learning", "section": "claim requirements",
        "text": "Learning claims: Annual limit 25,000. A receipt and manager approval are required. If either is missing, mark needs_review, not not_eligible. Claims above the limit are not eligible."
    },
    {
        "policy_id": "WELLNESS-ELIGIBILITY-v1", "benefit": "wellness", "section": "eligibility",
        "text": "Wellness: Only full-time and part-time employees in IN or US are eligible. Contractors are not eligible, regardless of claim purpose, receipt, or manager approval. Gym, yoga, and fitness expenses qualify for eligible employees; cosmetic treatments are excluded."
    },
    {
        "policy_id": "WELLNESS-CLAIM-v1", "benefit": "wellness", "section": "claim requirements",
        "text": "Wellness claims: Annual limit 12,000. A receipt is required; manager approval is not. Missing receipts need review. Claims above the limit are not eligible."
    },
    {
        "policy_id": "INTERNET-ELIGIBILITY-v1", "benefit": "internet", "section": "eligibility",
        "text": "Internet: Full-time and part-time employees in IN can claim home broadband. Employees outside IN and mobile-data-only plans are excluded."
    },
    {
        "policy_id": "INTERNET-CLAIM-v1", "benefit": "internet", "section": "claim requirements",
        "text": "Internet claims: Annual limit 6,000. A receipt is required; manager approval is not. Missing receipts need review. Claims above the limit are not eligible."
    },
]

POLICY_CONTEXT = "\n".join(f"[{p['policy_id']} | {p['benefit']} | {p['section']}] {p['text']}" for p in POLICY_PASSAGES)
print(f"Loaded policy text for {len({p['benefit'] for p in POLICY_PASSAGES})} benefits.")


Loaded policy text for 3 benefits.


## 5. Create the golden dataset

Each fixed case stores employee details and claim details together in one input object. Expected decisions are written explicitly so the benchmark stays the same across runs; have a policy owner verify them before real use.


In [13]:
GOLDEN_CASES = [{'case_id': 'BEN-001',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'eligible',
  'expected_reason': 'Eligible employee, relevant course, below limit, receipt and required approval '
                     'present.',
  'claim': {'benefit': 'learning',
            'amount': 12000,
            'purpose': 'job-related cloud certification',
            'receipt_provided': True,
            'manager_approved': True}},
 {'case_id': 'BEN-002',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'needs_review',
  'expected_reason': 'Learning claims require manager approval, which is missing.',
  'claim': {'benefit': 'learning',
            'amount': 5000,
            'purpose': 'job-related course',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-003',
  'employee': {'employment': 'contractor', 'location': 'IN'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'Contractors are not eligible for the learning allowance.',
  'claim': {'benefit': 'learning',
            'amount': 4000,
            'purpose': 'job-related course',
            'receipt_provided': True,
            'manager_approved': True}},
 {'case_id': 'BEN-004',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'Claim exceeds the 25,000 annual maximum.',
  'claim': {'benefit': 'learning',
            'amount': 26000,
            'purpose': 'job-related certification',
            'receipt_provided': True,
            'manager_approved': True}},
 {'case_id': 'BEN-005',
  'employee': {'employment': 'full_time', 'location': 'US'},
  'expected_decision': 'needs_review',
  'expected_reason': 'Required receipt is missing.',
  'claim': {'benefit': 'learning',
            'amount': 4000,
            'purpose': 'job-related course',
            'receipt_provided': False,
            'manager_approved': True}},
 {'case_id': 'BEN-006',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'Non-job-related courses are explicitly excluded.',
  'claim': {'benefit': 'learning',
            'amount': 3000,
            'purpose': 'unrelated hobby cooking course',
            'receipt_provided': True,
            'manager_approved': True}},
 {'case_id': 'BEN-007',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'eligible',
  'expected_reason': 'Gym membership is eligible; wellness does not require manager approval.',
  'claim': {'benefit': 'wellness',
            'amount': 4000,
            'purpose': 'gym membership',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-008',
  'employee': {'employment': 'part_time', 'location': 'US'},
  'expected_decision': 'eligible',
  'expected_reason': 'Part-time employees in the US are eligible; yoga is an eligible expense.',
  'claim': {'benefit': 'wellness',
            'amount': 9000,
            'purpose': 'yoga classes',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-009',
  'employee': {'employment': 'part_time', 'location': 'IN'},
  'expected_decision': 'needs_review',
  'expected_reason': 'Receipt is required but missing.',
  'claim': {'benefit': 'wellness',
            'amount': 5000,
            'purpose': 'fitness programme',
            'receipt_provided': False,
            'manager_approved': False}},
 {'case_id': 'BEN-010',
  'employee': {'employment': 'contractor', 'location': 'IN'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'Contractors are not eligible for wellness under the demo policy.',
  'claim': {'benefit': 'wellness',
            'amount': 3000,
            'purpose': 'gym membership',
            'receipt_provided': True,
            'manager_approved': True}},
 {'case_id': 'BEN-011',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'Claim exceeds the 12,000 annual maximum.',
  'claim': {'benefit': 'wellness',
            'amount': 14000,
            'purpose': 'gym membership',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-012',
  'employee': {'employment': 'full_time', 'location': 'US'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'Cosmetic treatments are explicitly excluded.',
  'claim': {'benefit': 'wellness',
            'amount': 2000,
            'purpose': 'cosmetic treatment',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-013',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'eligible',
  'expected_reason': 'Home broadband is eligible and no manager approval is required.',
  'claim': {'benefit': 'internet',
            'amount': 3000,
            'purpose': 'home fixed broadband',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-014',
  'employee': {'employment': 'part_time', 'location': 'IN'},
  'expected_decision': 'eligible',
  'expected_reason': 'Part-time employees in India are eligible for home internet.',
  'claim': {'benefit': 'internet',
            'amount': 4500,
            'purpose': 'home internet service',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-015',
  'employee': {'employment': 'full_time', 'location': 'US'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'The internet allowance is restricted to employees located in India.',
  'claim': {'benefit': 'internet',
            'amount': 3000,
            'purpose': 'home broadband',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-016',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'needs_review',
  'expected_reason': 'Required receipt is missing.',
  'claim': {'benefit': 'internet',
            'amount': 2000,
            'purpose': 'home broadband',
            'receipt_provided': False,
            'manager_approved': False}},
 {'case_id': 'BEN-017',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'Claim exceeds the 6,000 annual maximum.',
  'claim': {'benefit': 'internet',
            'amount': 7000,
            'purpose': 'home broadband',
            'receipt_provided': True,
            'manager_approved': False}},
 {'case_id': 'BEN-018',
  'employee': {'employment': 'full_time', 'location': 'IN'},
  'expected_decision': 'not_eligible',
  'expected_reason': 'Mobile data-only packages are excluded.',
  'claim': {'benefit': 'internet',
            'amount': 1500,
            'purpose': 'mobile data-only package',
            'receipt_provided': True,
            'manager_approved': False}}]

print("Golden cases:", len(GOLDEN_CASES))
print("Expected decisions:", {label: sum(r['expected_decision'] == label for r in GOLDEN_CASES) for label in ["eligible", "needs_review", "not_eligible"]})


Golden cases: 18
Expected decisions: {'eligible': 5, 'needs_review': 4, 'not_eligible': 9}


## 6. Inspect the golden dataset

Check the cases and expected labels before running the agent. This is just for a quick dataset sanity check ensuring everything is labelled and present properly.


In [14]:
golden_df = pd.DataFrame([
    {"case_id":r["case_id"], "benefit":r["claim"]["benefit"], "employment":r["employee"]["employment"],
     "location":r["employee"]["location"], "amount":r["claim"]["amount"],
     "receipt":r["claim"]["receipt_provided"], "manager_approved":r["claim"]["manager_approved"],
     "expected":r["expected_decision"], "purpose":r["claim"]["purpose"]}
    for r in GOLDEN_CASES
])
display(golden_df)

,case_id,benefit,employment,location,amount,receipt,manager_approved,expected,purpose
0,BEN-001,learning,full_time,IN,12000,True,True,eligible,job-related cloud certification
1,BEN-002,learning,full_time,IN,5000,True,False,needs_review,job-related course
2,BEN-003,learning,contractor,IN,4000,True,True,not_eligible,job-related course
3,BEN-004,learning,full_time,IN,26000,True,True,not_eligible,job-related certification
4,BEN-005,learning,full_time,US,4000,False,True,needs_review,job-related course
5,BEN-006,learning,full_time,IN,3000,True,True,not_eligible,unrelated hobby cooking course
6,BEN-007,wellness,full_time,IN,4000,True,False,eligible,gym membership
7,BEN-008,wellness,part_time,US,9000,True,False,eligible,yoga classes
8,BEN-009,wellness,part_time,IN,5000,False,False,needs_review,fitness programme
9,BEN-010,wellness,contractor,IN,3000,True,True,not_eligible,gym membership


## 7. Define the answer schema and V1 prompt

The LLM receives one input object with `employee` and `claim` fields, then returns one decision, a short explanation, supporting evidence, and any missing information. V1 is our starting prompt; we will use its errors to decide what to improve.


In [15]:
class BenefitDecision(BaseModel):
    decision: Literal["eligible", "not_eligible", "needs_review"] = Field(description="Recommended claim status")
    explanation: str = Field(description="Short explanation grounded in policy text")
    evidence: list[str] = Field(description="Relevant policy statements supporting the decision")
    missing_information: list[str] = Field(description="Information or documentation still needed, if any")

MODEL_NAME = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

# Simple starting prompt: it does not yet distinguish missing documentation from true ineligibility.
PROMPT_V1 = ChatPromptTemplate.from_messages([
    ("system", """You are an employee benefits assistant. Use only the policy text provided below to assess the claim. Return a structured decision. Do not invent policy rules. For this initial baseline, take a conservative approach to incomplete claims: if a required receipt is missing, return not_eligible. This is a deliberately limited starting rule to evaluate and improve against the supplied policy."""),
    ("human", "Employee and claim:\n{case_json}\n\nPolicy text:\n{policy_context}\n\nAssess this claim using the available policy information.")
])

print("Structured output schema and initial prompt are ready.")

Structured output schema and initial prompt are ready.


## 8. Run one smoke check

Run one representative case to confirm structured output, model invocation, and Langfuse tracing work before the full evaluation.


In [16]:
def make_agent(prompt, model=llm):
    return prompt | model.with_structured_output(BenefitDecision)

agent_v1 = make_agent(PROMPT_V1)

@observe(name="unused-benefits-agent")
def run_agent(case, version="v1"):
    case_id = case.get("case_id", "unknown")
    claim = case.get("claim", {})
    agent = agent_v2 if version.startswith("v2") else agent_v1
    case_json = json.dumps(
        {k: v for k, v in case.items() if not k.startswith("expected_")},
        ensure_ascii=False,
    )

    # Langfuse SDK v4 uses propagate_attributes for trace-level metadata.
    with propagate_attributes(
        trace_name=f"benefits-{version}-{case_id}",
        metadata={
            "case_id": case_id,
            "benefit": claim.get("benefit", "unknown"),
            "agent_version": version,
        },
    ):
        trace_id = langfuse.get_current_trace_id()
        response = agent.invoke(
            {"case_json": case_json, "policy_context": POLICY_CONTEXT},
            config={"callbacks": [CallbackHandler()]},
        )

    result = response.model_dump()
    result["agent_version"] = version
    result["trace_id"] = trace_id
    return result


SMOKE_CASE_ID = "BEN-001"
smoke_case = next((case for case in GOLDEN_CASES if case["case_id"] == SMOKE_CASE_ID), None)
if smoke_case is None:
    raise RuntimeError(f"Smoke-check case {SMOKE_CASE_ID!r} was not found in GOLDEN_CASES.")

smoke_output = run_agent(smoke_case, version="smoke")
print(json.dumps({
    "case_id": smoke_case["case_id"],
    "decision": smoke_output["decision"],
    "trace_id": smoke_output.get("trace_id", ""),
    "explanation": smoke_output["explanation"],
}, indent=2, ensure_ascii=False))
print("Smoke check completed. Open this trace in Langfuse to inspect it.")

{
  "case_id": "BEN-001",
  "decision": "eligible",
  "trace_id": "cf84fc4bdb7f1a43098a916579c77464",
  "explanation": "The claim meets all eligibility requirements, including receipt and manager approval, and is within the annual limit."
}
Smoke check completed. Open this trace in Langfuse to inspect it.


### Check the trace

In Langfuse, open the single smoke-check trace and inspect the claim input, policy text, model output, and timing.


## 9. Upload the golden dataset

Store the cases in Langfuse so the same benchmark can be reused for each experiment.


In [17]:
try:
    dataset = langfuse.get_dataset(DATASET_NAME)
    stored_items = list(getattr(dataset, "items", []) or [])
    stored_ids = [item.input.get("case_id") for item in stored_items if isinstance(item.input, dict)]
    expected_ids = [case["case_id"] for case in GOLDEN_CASES]
    if len(stored_items) != 18 or sorted(stored_ids) != sorted(expected_ids):
        raise RuntimeError("Existing golden dataset differs from the 18 local cases. Verify it before continuing.")
    print("Verified 18 existing golden dataset items.")
except Exception:
    langfuse.create_dataset(
        name=DATASET_NAME,
        description="Human-reviewable golden cases for a fictional benefits agent"
    )
    for case in GOLDEN_CASES:
        agent_input = {k:v for k,v in case.items() if not k.startswith("expected_") and k != "expected_decision"}
        langfuse.create_dataset_item(
            dataset_name=DATASET_NAME,
            input=agent_input,
            expected_output={"decision":case["expected_decision"]},
            metadata={"case_id":case["case_id"], "benefit":case["claim"]["benefit"], "expected_reason":case["expected_reason"]}
        )
    langfuse.flush()
    dataset = langfuse.get_dataset(DATASET_NAME)
    print(f"Uploaded {len(GOLDEN_CASES)} cases to Langfuse.")

Verified 18 existing golden dataset items.


## 10. Choose evaluation metrics

We measure decision accuracy, unsafe false-eligibility recommendations, and missed eligible claims.


In [18]:
def score_prediction(case, output):
    expected = case["expected_decision"]
    predicted = output["decision"]
    return {
        **case,
        "output":output,
        "predicted_decision":predicted,
        "correct":predicted == expected,
        "false_eligible":predicted == "eligible" and expected != "eligible",
        "missed_eligible":expected == "eligible" and predicted != "eligible",
    }

def summarize_results(rows, label):
    n = len(rows)
    if not n:
        return {"version":label,"cases":0}
    return {
        "version":label,
        "cases":n,
        "decision_accuracy":round(sum(r["correct"] for r in rows)/n,3),
        "false_eligible":sum(r["false_eligible"] for r in rows),
        "missed_eligible":sum(r["missed_eligible"] for r in rows),
    }

def exact_match_evaluator(*, output, expected_output, **kwargs):
    return Evaluation(name="decision_correct", value=int(output["decision"] == expected_output["decision"]))


## 11. Running V1 (baseline) : Evaluate all the cases present.

Run the agent on all golden cases. The case-level results and evaluation scores are recorded in Langfuse.


In [19]:
EVALUATION_OUTPUTS_BY_CASE = {}

def task_v1(item):
    output = run_agent(item.input, version="v1")
    case_id = item.input.get("case_id")
    if not case_id:
        raise ValueError("Dataset item input is missing case_id; recreate/sync the golden dataset before running.")
    EVALUATION_OUTPUTS_BY_CASE[case_id] = output
    return output

baseline_experiment = dataset.run_experiment(
    name=EXPERIMENT_V1_NAME,
    task=task_v1,
    evaluators=[exact_match_evaluator],
    metadata={"model": MODEL_NAME, "prompt_version": "v1", "policy_context": "direct_prompt"},
)
langfuse.flush()
print(f"Evaluation completed for {len(EVALUATION_OUTPUTS_BY_CASE)} cases.")


Evaluation completed for 18 cases.


## 12. Inspect the evaluation results

Review the decisions from the full evaluation - what v1 missed or falsely identified.


In [20]:
missing_outputs = [case["case_id"] for case in GOLDEN_CASES if case["case_id"] not in EVALUATION_OUTPUTS_BY_CASE]
if missing_outputs:
    raise RuntimeError(
        "The evaluation did not produce outputs for these cases: "
        + ", ".join(missing_outputs)
        + ". Check the Langfuse dataset and rerun the evaluation cell."
    )

baseline_rows = [
    score_prediction(case, EVALUATION_OUTPUTS_BY_CASE[case["case_id"]])
    for case in GOLDEN_CASES
]
print(summarize_results(baseline_rows, "Initial evaluation"))

{'version': 'Initial evaluation', 'cases': 18, 'decision_accuracy': 0.556, 'false_eligible': 0, 'missed_eligible': 4}


The table below shows the decisions from the evaluation run, including which cases matched or disagreed with the expected decision.


In [21]:
failures_v1 = [r for r in baseline_rows if not r["correct"]]
print("Decision failures to review:", len(failures_v1))
for row in failures_v1:
    print("\n", row["case_id"], row["claim"]["benefit"],
          "expected=", row["expected_decision"],
          "predicted=", row["predicted_decision"])
    print("Purpose:", row["claim"]["purpose"])
    print("Agent explanation:", row["output"].get("explanation"))

Decision failures to review: 8

 BEN-002 learning expected= needs_review predicted= not_eligible
Purpose: job-related course
Agent explanation: The claim is not eligible because manager approval is required and it has not been provided.

 BEN-005 learning expected= needs_review predicted= not_eligible
Purpose: job-related course
Agent explanation: The claim is not eligible because a required receipt is missing.

 BEN-007 wellness expected= eligible predicted= not_eligible
Purpose: gym membership
Agent explanation: The claim exceeds the annual limit for wellness claims, which is 12,000.

 BEN-008 wellness expected= eligible predicted= not_eligible
Purpose: yoga classes
Agent explanation: The claim is not eligible because manager approval is required for wellness claims, and it was not provided.

 BEN-009 wellness expected= needs_review predicted= not_eligible
Purpose: fitness programme
Agent explanation: The claim is not eligible because a receipt is required and none was provided.

 BE

Next, we select few genuine failures and genuine successes cases for human review.

In [23]:
def get_paginated_data(endpoint):
    page = 1
    found = []

    while True:
        response = requests.get(
            f"{BASE_URL}/api/public/{endpoint}",
            auth=AUTH,
            params={"page": page, "limit": 100},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        found.extend(payload.get("data") or [])

        meta = payload.get("meta") or {}
        if page >= meta.get("totalPages", page):
            return found

        page += 1


def find_unique_by_name(endpoint, name, item_type):
    matches = [
        item for item in get_paginated_data(endpoint)
        if item.get("name") == name
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {item_type} named {name!r}; "
            f"found {len(matches)}. Check Langfuse."
        )

    return matches[0]


# Verify both score configurations and the existing queue.
decision_correct_config = find_unique_by_name(
    "score-configs", SCORE_CONFIG_NAME, "Score Configuration"
)
human_decision_config = find_unique_by_name(
    "score-configs", HUMAN_SCORE_CONFIG_NAME, "Score Configuration"
)
queue = find_unique_by_name(
    "annotation-queues", ANNOTATION_QUEUE_NAME, "Annotation Queue"
)

QUEUE_ID = queue["id"]
SCORE_CONFIG_ID = decision_correct_config["id"]
HUMAN_SCORE_CONFIG_ID = human_decision_config["id"]

print(f"Existing queue: {ANNOTATION_QUEUE_NAME}")
print(f"Queue ID: {QUEUE_ID}")
print(f"Queue score configuration IDs: {queue.get('scoreConfigIds', [])}")
print(f"decision_correct ID: {SCORE_CONFIG_ID}")
print(f"human_decision ID: {HUMAN_SCORE_CONFIG_ID}")

if not {HUMAN_SCORE_CONFIG_ID, SCORE_CONFIG_ID}.issubset(set(queue.get("scoreConfigIds") or [])):
    raise RuntimeError(
        "The existing queue is not configured with human_decision. "
        "Add that categorical Score Configuration to the queue in "
        "Langfuse before continuing. This cell will not modify the queue."
    )


# Select five genuine failures and five genuine successes.
failures_v1 = [row for row in baseline_rows if not row["correct"]]
successes_v1 = [row for row in baseline_rows if row["correct"]]

selected_failures = failures_v1[:5]
selected_successes = successes_v1[:5]

if len(selected_failures) != 5 or len(selected_successes) != 5:
    raise RuntimeError(
        f"Expected at least five failures and five successes; found "
        f"{len(failures_v1)} failures and {len(successes_v1)} successes."
    )

review_rows = selected_failures + selected_successes

review_df = pd.DataFrame([
    {
        "case_id": row["case_id"],
        "review_group": (
            "incorrect_prediction" if not row["correct"]
            else "correct_prediction"
        ),
        "benefit": row["claim"]["benefit"],
        "expected_decision": row["expected_decision"],
        "agent_decision": row["predicted_decision"],
        "trace_id": row["output"].get("trace_id", ""),
        "agent_explanation": row["output"].get("explanation", ""),
    }
    for row in review_rows
])


# Create the review dataset if needed; otherwise reuse it.
try:
    langfuse.get_dataset(REVIEW_DATASET_NAME)
    print(f"Reusing dataset: {REVIEW_DATASET_NAME}")
except Exception:
    langfuse.create_dataset(
        name=REVIEW_DATASET_NAME,
        description=(
            "Ten employee-benefit decisions: five incorrect and "
            "five correct agent predictions for human review."
        ),
    )
    print(f"Created dataset: {REVIEW_DATASET_NAME}")


# Fetch existing queue items so we don't deliberately add duplicates.
queue_items = get_paginated_data(
    f"annotation-queues/{QUEUE_ID}/items"
)

# Start each rehearsal with an empty queue, preserving its configuration.
if queue_items:
    raise RuntimeError(f"Queue has {len(queue_items)} old items. Clear them in Langfuse before running this cell.")
existing_trace_ids = {
    item.get("objectId")
    for item in queue_items
    if item.get("objectId")
}

queued = 0
already_queued = 0

for row in review_rows:
    trace_id = row["output"].get("trace_id")

    if not trace_id:
        raise RuntimeError(
            f"No trace ID returned for {row['case_id']}."
        )

    # Reuse dataset items if they already exist; avoid blindly duplicating.
    try:
        langfuse.create_dataset_item(
            dataset_name=REVIEW_DATASET_NAME,
            input={
                "case_id": row["case_id"],
                "employee": row["employee"],
                "claim": row["claim"],
            },
            expected_output={"decision": row["expected_decision"]},
            metadata={
                "case_id": row["case_id"],
                "review_group": (
                    "incorrect_prediction" if not row["correct"]
                    else "correct_prediction"
                ),
                "agent_decision": row["predicted_decision"],
            },
        )
    except Exception as exc:
        print(
            f"Dataset item for {row['case_id']} was not added "
            f"(it may already exist): {exc}"
        )

    if trace_id in existing_trace_ids:
        already_queued += 1
        print(f"{row['case_id']} already exists in the queue.")
        continue

    response = requests.post(
        f"{BASE_URL}/api/public/annotation-queues/{QUEUE_ID}/items",
        auth=AUTH,
        json={
            "objectId": trace_id,
            "objectType": "TRACE",
            "status": "PENDING",
        },
        timeout=30,
    )

    if response.status_code in (200, 201):
        queued += 1
        existing_trace_ids.add(trace_id)
    elif response.status_code == 409:
        already_queued += 1
        print(f"{row['case_id']} is already in the queue.")
    else:
        print(
            f"Failed to queue {row['case_id']}: "
            f"{response.status_code} {response.text}"
        )
        response.raise_for_status()

langfuse.flush()

print(
    f"\nReview sample: {len(selected_failures)} failures + "
    f"{len(selected_successes)} successes."
)
print(f"Newly queued: {queued}")
print(f"Already in queue: {already_queued}")

display(review_df)


Existing queue: unused-benefits-v1-human-review
Queue ID: cmvfl7ffc00zsad0jzlug41aa
Queue score configuration IDs: ['cmvfl7cig03e3ad0e7ckk89xt', 'cmvfnd69d00ckad0jj662eesh']
decision_correct ID: cmvfl7cig03e3ad0e7ckk89xt
human_decision ID: cmvfnd69d00ckad0jj662eesh
Reusing dataset: benefits-20261020-112631-review

Review sample: 5 failures + 5 successes.
Newly queued: 10
Already in queue: 0


,case_id,review_group,benefit,expected_decision,agent_decision,trace_id,agent_explanation
0,BEN-002,incorrect_prediction,learning,needs_review,not_eligible,2d6be909aee9548d3c07ea26fc1787fa,The claim is not eligible because manager appr...
1,BEN-005,incorrect_prediction,learning,needs_review,not_eligible,89b35787924c9906f48f389d3c43a771,The claim is not eligible because a required r...
2,BEN-007,incorrect_prediction,wellness,eligible,not_eligible,6554d6ca249230a6a35d748196f64266,The claim exceeds the annual limit for wellnes...
3,BEN-008,incorrect_prediction,wellness,eligible,not_eligible,b0cade3c292ba85fc8277148794b717f,The claim is not eligible because manager appr...
4,BEN-009,incorrect_prediction,wellness,needs_review,not_eligible,f579f6b74536b34dae9b19c50e14373d,The claim is not eligible because a receipt is...
5,BEN-001,correct_prediction,learning,eligible,eligible,d06ec5ef7a98c7967eb5e24a4bb769da,"The claim meets all eligibility requirements, ..."
6,BEN-003,correct_prediction,learning,not_eligible,not_eligible,d00f94841cfe673f41847b4bb425b046,Contractors are not eligible for learning bene...
7,BEN-004,correct_prediction,learning,not_eligible,not_eligible,96428b5b2d0cd5e80446e24fa4897b60,The claim amount exceeds the annual limit of 2...
8,BEN-006,correct_prediction,learning,not_eligible,not_eligible,b0d342ede5337d539ff0458b31927e40,The course is unrelated to the employee's job ...
9,BEN-010,correct_prediction,wellness,not_eligible,not_eligible,ed354e5ccd43d2472f47e5153eea8c7c,Contractors are not eligible for wellness clai...


**Pause for review.** Open the queue named in `ANNOTATION_QUEUE_NAME` in Langfuse. Complete the ten new cases, recording both `decision_correct` and `human_decision`. Then run the next cell to fetch human scores. Do not reuse annotations from earlier runs.

## 15. Fetch the saved human decisions

Retrieve the `human_decision` scores saved in Langfuse and match them to the reviewed traces.


In [24]:
# Fetch existing human scores and match them to the selected review traces.

def _as_dict(obj):
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump(by_alias=True)
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    return getattr(obj, "__dict__", {})


human_scores = []
cursor = None

while True:
    params = {
        "name": "human_decision",
        "limit": 100,
        "fields": "core,subject,annotation",
    }
    if cursor:
        params["cursor"] = cursor

    response = requests.get(
        f"{BASE_URL}/api/public/v3/scores",
        auth=AUTH,
        params=params,
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    human_scores.extend(payload.get("data", []) or [])

    meta = payload.get("meta") or {}
    next_cursor = meta.get("cursor")

    if not next_cursor or next_cursor == cursor:
        break

    cursor = next_cursor


# Match human decisions to the selected review traces.
trace_to_case = {
    row["output"].get("trace_id"): row
    for row in review_rows
    if row["output"].get("trace_id")
}

human_by_case = {}

for raw in human_scores:
    score = _as_dict(raw)

    if score.get("name") != "human_decision":
        continue

    subject = score.get("subject") or {}
    trace_id = (
        subject.get("id")
        if subject.get("kind") == "trace"
        else None
    )
    
    row = trace_to_case.get(trace_id)
    decision = str(score.get("value", "")).strip().lower()

    if row and decision in {
        "eligible",
        "not_eligible",
        "needs_review",
    }:
        human_by_case[row["case_id"]] = {
            "human_decision": decision,
            "human_notes": str(
                score.get("comment")
                or score.get("stringValue")
                or ""
            ),
        }


# Build the calibration table.
calibration_df = pd.DataFrame([
    {
        "case_id": row["case_id"],
        "review_group": (
            "incorrect_prediction"
            if not row["correct"]
            else "correct_prediction"
        ),
        "reference_decision": row["expected_decision"],
        "agent_decision": row["predicted_decision"],
        "human_decision": human_by_case.get(
            row["case_id"], {}
        ).get("human_decision", ""),
        "human_notes": human_by_case.get(
            row["case_id"], {}
        ).get("human_notes", ""),
        "human_verified": row["case_id"] in human_by_case,
        "jev_choice": "",
    }
    for row in review_rows
])

print(
    f"Fetched {len(human_by_case)} of "
    f"{len(review_rows)} human decisions."
)

if len(human_by_case) < len(review_rows):
    print(
        "Some decisions are not yet visible as scores. "
        "Check that the reviews were submitted with the score name "
        "'human_decision' and that each score uses one of the expected "
        "decision values."
    )

display(calibration_df)


Fetched 10 of 10 human decisions.


,case_id,review_group,reference_decision,agent_decision,human_decision,human_notes,human_verified,jev_choice
0,BEN-002,incorrect_prediction,needs_review,not_eligible,needs_review,,True,
1,BEN-005,incorrect_prediction,needs_review,not_eligible,needs_review,,True,
2,BEN-007,incorrect_prediction,eligible,not_eligible,eligible,,True,
3,BEN-008,incorrect_prediction,eligible,not_eligible,eligible,,True,
4,BEN-009,incorrect_prediction,needs_review,not_eligible,needs_review,,True,
5,BEN-001,correct_prediction,eligible,eligible,eligible,,True,
6,BEN-003,correct_prediction,not_eligible,not_eligible,not_eligible,,True,
7,BEN-004,correct_prediction,not_eligible,not_eligible,not_eligible,,True,
8,BEN-006,correct_prediction,not_eligible,not_eligible,not_eligible,,True,
9,BEN-010,correct_prediction,not_eligible,not_eligible,not_eligible,,True,


## 16. Setting up v2: Improving the prompt

Use the evaluation results and human review notes to make policy checks more explicit, especially exclusions, claim limits, and missing requirements. Keep the policy and test cases fixed so the comparison remains fair.


In [25]:
PROMPT_V2 = ChatPromptTemplate.from_messages([
    ("system", """
You are an employee benefits assistant. Evaluate claims using only the supplied company policies.

For the requested benefit, check each condition before deciding:
1. Employment type and location must be allowed.
2. Expense purpose must be covered.
3. Claim amount must not exceed the benefit's exact annual limit.
4. A receipt must be provided.
5. Manager approval is required only when the benefit policy explicitly says so.

Apply these decision rules in order:
- If employment, location, purpose, or amount violates the policy, return not_eligible.
- Otherwise, if a required receipt or manager approval is missing, return needs_review.
- Otherwise, return eligible.
- If necessary policy information is unavailable, return needs_review.
For claim limits, compare the numeric amounts explicitly.
A claim is within the limit only when claim_amount <= annual_limit.
If claim_amount > annual_limit, return not_eligible.

For example, 7000 > 6000, so a claim of 7000
against a limit of 6000 is not eligible.

Never describe an amount above the limit as within the limit.

Never apply rules from another benefit.
Never invent a policy limit or restriction.
Never reject part-time employees when the policy includes them.
Never require manager approval when the policy says it is unnecessary.

Before returning your answer, verify that the decision and explanation agree with every applicable policy condition.

Return the structured decision and a brief explanation.
"""),
    ("human", """
Company policies:
{policy_context}

Employee claim:
{case_json}
""")
])


agent_v2 = make_agent(PROMPT_V2)
print("V2 prompt is ready. No golden labels or cases were changed.")


V2 prompt is ready. No golden labels or cases were changed.


## 17. Evaluation of v2 Experiment

Run the revised prompt against the same cases from the golden dataset with same metrics.


In [26]:
# Record V2 outputs from this experiment so the comparison table reuses the same predictions.
V2_OUTPUTS_BY_CASE = {}

def task_v2(item):
    output = run_agent(item.input, version="v2")
    case_id = item.input.get("case_id")
    if not case_id:
        raise ValueError("Dataset item input is missing case_id; recreate/sync the golden dataset before running.")
    V2_OUTPUTS_BY_CASE[case_id] = output
    return output

v2_experiment = dataset.run_experiment(
    name=EXPERIMENT_V2_NAME,
    task=task_v2,
    evaluators=[exact_match_evaluator],
    metadata={"model":MODEL_NAME, "prompt_version":"v2", "policy_context":"direct_prompt", "base_version":"v1"}
)
langfuse.flush()
print(f"V2 experiment completed. Recorded outputs for {len(V2_OUTPUTS_BY_CASE)} case(s).")


V2 experiment completed. Recorded outputs for 18 case(s).


## 18. Compare V1 and V2

Inspect overall metrics and case-level changes. Check which failures improved, which stayed wrong, and whether any previously correct cases regressed.

In [27]:
# Use the same V2 experiment outputs for local comparison; do not rerun the model.
missing_v2_outputs = [case["case_id"] for case in GOLDEN_CASES if case["case_id"] not in V2_OUTPUTS_BY_CASE]
if missing_v2_outputs:
    raise RuntimeError(
        "The V2 experiment did not produce outputs for these golden cases: "
        + ", ".join(missing_v2_outputs)
        + ". Check the Langfuse dataset and rerun the V2 experiment cell."
    )

v2_rows = [
    score_prediction(case, V2_OUTPUTS_BY_CASE[case["case_id"]])
    for case in GOLDEN_CASES
]
comparison = pd.DataFrame([
    {"case_id":a["case_id"], "benefit":a["claim"]["benefit"], "expected":a["expected_decision"],
     "v1":a["predicted_decision"], "v2":b["predicted_decision"],
     "v1_correct":a["correct"], "v2_correct":b["correct"]}
    for a,b in zip(baseline_rows, v2_rows)
])
print(summarize_results(baseline_rows, "V1"))
print(summarize_results(v2_rows, "V2"))
print("Wins:", int(((~comparison["v1_correct"]) & comparison["v2_correct"]).sum()))
print("Regressions:", int((comparison["v1_correct"] & (~comparison["v2_correct"])).sum()))
display(comparison)


{'version': 'V1', 'cases': 18, 'decision_accuracy': 0.556, 'false_eligible': 0, 'missed_eligible': 4}
{'version': 'V2', 'cases': 18, 'decision_accuracy': 1.0, 'false_eligible': 0, 'missed_eligible': 0}
Wins: 8
Regressions: 0


,case_id,benefit,expected,v1,v2,v1_correct,v2_correct
0,BEN-001,learning,eligible,eligible,eligible,True,True
1,BEN-002,learning,needs_review,not_eligible,needs_review,False,True
2,BEN-003,learning,not_eligible,not_eligible,not_eligible,True,True
3,BEN-004,learning,not_eligible,not_eligible,not_eligible,True,True
4,BEN-005,learning,needs_review,not_eligible,needs_review,False,True
5,BEN-006,learning,not_eligible,not_eligible,not_eligible,True,True
6,BEN-007,wellness,eligible,not_eligible,eligible,False,True
7,BEN-008,wellness,eligible,not_eligible,eligible,False,True
8,BEN-009,wellness,needs_review,not_eligible,needs_review,False,True
9,BEN-010,wellness,not_eligible,not_eligible,not_eligible,True,True


## 19. Save known failure cases for regression checks

Keep the incorrect cases as a focused regression set so future prompt changes can be checked against the errors that motivated the improvements.


In [28]:
regression_cases = failures_v1
print("V1 failure cases available for regression:", len(regression_cases))

try:
    regression_dataset = langfuse.get_dataset(REGRESSION_DATASET_NAME)
    print(f"Dataset {REGRESSION_DATASET_NAME!r} already exists; reusing it.")
except Exception:
    langfuse.create_dataset(
        name=REGRESSION_DATASET_NAME,
        description="Known V1 failures for the fictional benefits agent"
    )
    for row in regression_cases:
        agent_input = {k:v for k,v in row.items() if k in ["case_id","employee","claim"]}
        langfuse.create_dataset_item(
            dataset_name=REGRESSION_DATASET_NAME,
            input=agent_input,
            expected_output={"decision":row["expected_decision"]},
            metadata={"case_id":row["case_id"], "source":"v1_failure", "human_review_required":True}
        )
    langfuse.flush()
    regression_dataset = langfuse.get_dataset(REGRESSION_DATASET_NAME)
    print("Regression dataset created.")


V1 failure cases available for regression: 8
Regression dataset created.


## 20. Run the regression experiment

Run the updated prompt on the previously incorrect cases to check whether the targeted fixes worked.


In [29]:
if regression_cases:
    regression_experiment = regression_dataset.run_experiment(
        name=f"{RUN_LABEL}-v2-regression",
        task=task_v2,
        evaluators=[exact_match_evaluator],
        metadata={"model":MODEL_NAME, "prompt_version":"v2", "source":"v1_failures"}
    )
    langfuse.flush()
    print("Regression experiment submitted to Langfuse.")
else:
    print("No V1 failures to replay yet. Review and add verified edge cases before relying on regression testing.")


Regression experiment submitted to Langfuse.


## 21. Evaluate with Jev

**Pause for evaluator execution.** In Langfuse, go to **Experiments → Run Evaluator** for the current `EXPERIMENT_V1_NAME` and `EXPERIMENT_V2_NAME`. Execute the saved Jev evaluator once for each. Then fetch scores below.

In [32]:
# Use the exact score name configured in Langfuse.
jev_scores = []
cursor = None

while True:
    params = {
        "name": JEV_SCORE_NAME,
        "fields": "core,subject,annotation",
        "limit": 100,
    }

    if cursor:
        params["cursor"] = cursor

    response = requests.get(
        f"{BASE_URL}/api/public/v3/scores",
        auth=AUTH,
        params=params,
        timeout=30,
    )
    response.raise_for_status()

    payload = response.json()
    jev_scores.extend(payload.get("data", []))

    next_cursor = (payload.get("meta") or {}).get("cursor")

    if not next_cursor or next_cursor == cursor:
        break

    cursor = next_cursor

print(f"Retrieved {len(jev_scores)} Jev scores.")


Retrieved 72 Jev scores.


## 22. Compare Jev with the verified human reference

Compare Jev with human labels only when both a human label and an actual saved Jev result are available.


In [33]:
# Match Jev evaluations using case ID and experiment version.

VALID = {"eligible", "not_eligible", "needs_review"}

# Human-reviewed decisions already saved in calibration_df.
human_reference = calibration_df.set_index("case_id")[
    "human_decision"
].to_dict()

# Golden answers from the existing dataset.
golden_reference = {
    row["case_id"]: row["expected_decision"]
    for row in baseline_rows
}

# Read experiment metadata for each Jev-evaluated trace.
trace_cache = {}
results = []

for score in jev_scores:
    if score.get("name") != "based_employee_details":
        continue

    subject = score.get("subject") or {}
    trace_id = (
        subject.get("traceId")
        if subject.get("kind") == "observation"
        else subject.get("id")
    )

    if not trace_id:
        continue

    if trace_id not in trace_cache:
        response = requests.get(
            f"{BASE_URL}/api/public/v2/observations",
            auth=AUTH,
            params={
                "traceId": trace_id,
                "fields": "core,basic,io,metadata,trace_context",
                "limit": 100,
            },
            timeout=30,
        )
        response.raise_for_status()

        trace_cache[trace_id] = response.json().get("data", [])

    tasks = [
        obs for obs in trace_cache[trace_id]
        if obs.get("name") == "experiment-item-task"
    ]

    if len(tasks) != 1:
        continue

    metadata = tasks[0].get("metadata") or {}
    case_id = metadata.get("case_id")
    version = metadata.get("prompt_version")

    if metadata.get("experiment_name") not in {EXPERIMENT_V1_NAME, EXPERIMENT_V2_NAME}:
        continue
    decision = str(score.get("value", "")).strip().lower()

    if (
        case_id in golden_reference
        and version in {"v1", "v2"}
        and decision in VALID
    ):
        results.append({
            "case_id": case_id,
            "version": version,
            "jev_decision": decision,
            "golden_decision": golden_reference[case_id],
            "trace_id": trace_id,
            "score_id": score.get("id"),
            "timestamp": score.get("timestamp"),
        })

jev_results_df = pd.DataFrame(results)

print("Jev scores mapped:", len(jev_results_df))
print("By version:", dict(Counter(
    r["version"] for r in results
)))

# Prevent duplicate scores from silently overriding one another.
if not jev_results_df.empty:
    duplicates = jev_results_df.duplicated(
        ["case_id", "version"], keep=False
    )
    if duplicates.any():
        print("Multiple Jev scores found for some cases.")
        display(jev_results_df.loc[duplicates])
        raise RuntimeError(
            "Resolve duplicate case/version scores before calibration."
        )

# V2: Jev against the golden dataset.
v2_jev = jev_results_df[
    jev_results_df["version"] == "v2"
].copy()

if not v2_jev.empty:
    v2_jev["correct"] = (
        v2_jev["jev_decision"] == v2_jev["golden_decision"]
    )

print("\nV2 Jev coverage:", len(v2_jev), "/ 18")

if len(v2_jev) == 18:
    print("Jev vs golden accuracy:",
          f"{v2_jev['correct'].mean():.1%}")
else:
    print("V2 evaluation coverage is incomplete.")

# V1: Jev against the ten human-reviewed decisions.
reviewed_ids = set(calibration_df.loc[
    calibration_df["human_verified"].astype(bool)
    & calibration_df["human_decision"].isin(VALID),
    "case_id"
])

v1_jev = jev_results_df[
    (jev_results_df["version"] == "v1")
    & (jev_results_df["case_id"].isin(reviewed_ids))
].copy()

v1_jev["human_decision"] = (
    v1_jev["case_id"].map(human_reference)
)

if not v1_jev.empty:
    v1_jev["agreement"] = (
        v1_jev["jev_decision"] == v1_jev["human_decision"]
    )

print("\nHuman/Jev coverage:", len(v1_jev), "/ 10")

if len(v1_jev) == 10:
    print("Human/Jev agreement:",
          f"{v1_jev['agreement'].mean():.1%}")
else:
    print("Human calibration coverage is incomplete.")

display(v1_jev[[
    "case_id", "human_decision",
    "jev_decision", "agreement"
]] if not v1_jev.empty else v1_jev)

display(v2_jev[[
    "case_id", "golden_decision",
    "jev_decision", "correct"
]] if not v2_jev.empty else v2_jev)


Jev scores mapped: 36
By version: {'v2': 18, 'v1': 18}

V2 Jev coverage: 18 / 18
Jev vs golden accuracy: 100.0%

Human/Jev coverage: 10 / 10
Human/Jev agreement: 80.0%


,case_id,human_decision,jev_decision,agreement
20,BEN-010,not_eligible,not_eligible,True
24,BEN-008,eligible,eligible,True
27,BEN-003,not_eligible,not_eligible,True
28,BEN-009,needs_review,not_eligible,False
30,BEN-004,not_eligible,not_eligible,True
31,BEN-001,eligible,eligible,True
32,BEN-002,needs_review,needs_review,True
33,BEN-005,needs_review,needs_review,True
34,BEN-007,eligible,needs_review,False
35,BEN-006,not_eligible,not_eligible,True


,case_id,golden_decision,jev_decision,correct
0,BEN-003,not_eligible,not_eligible,True
1,BEN-009,needs_review,needs_review,True
2,BEN-017,not_eligible,not_eligible,True
3,BEN-014,eligible,eligible,True
4,BEN-013,eligible,eligible,True
5,BEN-018,not_eligible,not_eligible,True
6,BEN-010,not_eligible,not_eligible,True
7,BEN-008,eligible,eligible,True
8,BEN-015,not_eligible,not_eligible,True
9,BEN-011,not_eligible,not_eligible,True


## 23. Final checks

Report V2 accuracy and known-case regression coverage separately from Jev calibration. Jev calibration is complete when all ten human-reviewed cases and all 18 V2 cases have real matched scores; agreement does not imply production readiness.


In [34]:
final_summary = summarize_results(v2_rows, "V2")

known_regression_rows = [
    r for r in v2_rows
    if r["case_id"] in {x["case_id"] for x in regression_cases}
]

regression_clean = all(r["correct"] for r in known_regression_rows)

agent_pass = (
    final_summary.get("decision_accuracy", 0) >= 0.90
    and final_summary.get("false_eligible", 0) == 0
    and regression_clean
)

print("Final V2 summary:", final_summary)
print(
    "Known V1 failures now correct:",
    sum(r["correct"] for r in known_regression_rows),
    "/", len(known_regression_rows)
)
print("Agent teaching gate:", "PASS" if agent_pass else "FAIL")

# Jev calibration and evaluation
jev_gate = (
    len(v1_jev) == 10
    and len(v2_jev) == 18
    and v1_jev["human_decision"].notna().all()
    and v1_jev["jev_decision"].notna().all()
    and v2_jev["jev_decision"].notna().all()
)

print(
    "Jev calibration:",
    "COMPLETE" if jev_gate else "PENDING"
)

if jev_gate:
    print(
        "Human/Jev agreement:",
        f"{v1_jev['agreement'].mean():.1%}"
    )
    print(
        "Jev/Golden accuracy:",
        f"{v2_jev['correct'].mean():.1%}"
    )
    print(
        "Jev disagreements:",
        (~v2_jev["correct"]).sum()
    )


Final V2 summary: {'version': 'V2', 'cases': 18, 'decision_accuracy': 1.0, 'false_eligible': 0, 'missed_eligible': 0}
Known V1 failures now correct: 8 / 8
Agent teaching gate: PASS
Jev calibration: COMPLETE
Human/Jev agreement: 80.0%
Jev/Golden accuracy: 100.0%
Jev disagreements: 0
